# First Quantized Hamiltonian

SELECT and PREPARE for the first quantized chemistry Hamiltonian.

Here we assume the Born-Oppenheimer Hamiltonian and seek to simulate a
collection of $\eta$ electrons and $L$ static nuclei with a Hamiltonian given
by:
$$
H_{BO} = T + U + V + \frac{1}{2}
\sum_{\ell\ne\kappa=1}^L\frac{\zeta_\ell\zeta_\kappa}{\lVert R_\ell - R_\kappa \rVert}
$$

In the first quantized approach we assume periodic boundary conditions and use a
plane wave Galerkin discretization.
A plane wave basis function is given by
$$
\phi_p(r) = \frac{1}{\sqrt{\Omega}} e^{-i k_p\cdot r}
$$
where $r$ is a position vector in real space, $\Omega$ is the simulation cell
volume and $k_p$ is a reciprocal lattice vector.
In three dimensions we have
$$
k_p = \frac{2\pi p }{\Omega}
$$
for $p \in G$ and
$$
G = [-\frac{N^{1/3} -
1}{2},\frac{N^{1/3} - 1}{2}]^3 \subset \mathcal{Z}^3.
$$
and $N$ is the total number of planewaves.

With these definitions we can write the components of the Hamiltonian as:
$$
T = \sum_{i}^\eta\sum_{p\in G}\frac{\lVert k_p\rVert^2}{2} |p\rangle\langle p|_i
$$
which defines the kinetic energy of the electrons,
$$
U = -\frac{4\pi}{\Omega}
\sum_{\ell=1}^L \sum_{i}^\eta
\sum_{p,q\in G, p\ne q}
\left(
    \zeta_{\ell}
    \frac{e^{i k_{q-p}\cdot R_\ell}}{\lVert k_{p-q}\rVert^2}
    |p\rangle\langle q|_i
\right)
$$
describes the interaction of the electrons and the nuclei, and,
$$
V = \frac{2\pi}{\Omega}
\sum_{i\ne j=1}^\eta
\sum_{p,q\in G, p\ne q}
\sum_{\nu \in G_0}
\left(
    \frac{1}{\lVert k_{\nu}\rVert^2}
    |p + \nu\rangle\langle p|_i
    |q -\nu\rangle\langle q|_i
\right)
$$
describes the electron-electron interaction. The notation $|p\rangle\langle p|_i$ is shorthand for
$I_1\otimes\cdots\otimes |p\rangle \langle p |_j \otimes \cdots \otimes I_\eta$.
The system is represented using a set of $\eta$ signed integer registers each of
size $3 n_p$ where $n_p =  \lceil \log (N^{1/3} + 1) \rceil$, with the factor of
3 accounting for the 3 spatial dimensions.

In the first quantized approach, fermion antisymmetry is encoded through initial
state preparation. Spin labels are also absent and should be accounted for
during state preparation. The cost of initial state preparation is typically
ignored.
Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

The seven examples reproduce the original resource queries. Their
signatures and call graphs are declarations. Several source leaves have
no decomposition. The counts shown here are logical estimates.


In [ ]:
from dataclasses import asdict

from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.first_quantization_catalog import notebook_cases
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.qec.resources.estimation import estimate_ftqc_resources

cases = notebook_cases()

## `PrepareFirstQuantization`
State preparation for the first quantized chemistry Hamiltonian.

Prepares the state in Eq. 48 of the reference.

#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons.
 - `num_atoms`: The number of atoms. $L$ in the reference.
 - `lambda_zeta`: sum of nuclear charges.
 - `m_param`: $\mathcal{M}$ in the reference.
 - `num_bits_nuc_pos`: The number of bits of precision for representing the nuclear coordinates.
 - `num_bits_t`: The number of bits of precision for the state preparation over the register selecting between the different components of the Hamiltonian.
 - `num_bits_rot_aa`: The number of bits of precision for the rotation for amplitude amplification.
 - `sum_of_l1_coeffs`: The one-norm of the Hamiltonian coefficients to prepare (often called lambda in the literature.) 

#### Registers
 - `tuv`: Flag register for selecting between kinetic and potential terms in the Hamiltonian.
 - `uv`: Flag register for selecting between the different potential components of the Hamiltonian.
 - `i`: A register for selecting electronic registers.
 - `j`: A register for selecting electronic registers.
 - `w`: A register for selecting x, y and z components of the momentum register.
 - `r`: A register for controlling elements of the momentum register. Used for block encodiding kinetic energy operator.
 - `s`: A register for controlling elements of the momentum register. Used for block encodiding kinetic energy operator.
 - `mu`: A register used for implementing nested boxes for the momentum state preparation.
 - `nu_x`: x component of the momentum register for Coulomb potential.
 - `nu_y`: y component of the momentum register for Coulomb potential.
 - `nu_z`: z component of the momentum register for Coulomb potential.
 - `m`: an ancilla register in a uniform superposition.
 - `l`: The register for selecting the nuclei. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). 


In [ ]:
signature, model = cases["prepare"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)

## `SelectFirstQuantization`
SELECT operation for the first quantized chemistry Hamiltonian.

#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons.
 - `num_atoms`: The number of atoms. $L$ in the reference.
 - `lambda_zeta`: sum of nuclear charges.
 - `m_param`: $\mathcal{M}$ in the reference.
 - `num_bits_nuc_pos`: The number of bits of precision for representing the nuclear coordinates.
 - `num_bits_t`: The number of bits of precision for the state preparation over the register selecting between the different components of the Hamiltonian.
 - `num_bits_rot_aa`: The number of bits of precision for the rotation for amplitude amplification. 

#### Registers
 - `tuv`: Flag register for selecting between kinetic and potential terms in the Hamiltonian.
 - `uv`: Flag register for selecting between the different potential components of the Hamiltonian.
 - `i_ne_j`: Register flagging $i \ne j$
 - `plus_t`: A register prepared in the $|+\rangle$ state.
 - `i`: A register for selecting electronic registers.
 - `j`: A register for selecting electronic registers.
 - `w`: A register for selecting x, y and z components of the momentum register.
 - `r`: A register for controlling elements of the momentum register. Used for block encodiding kinetic energy operator.
 - `s`: A register for controlling elements of the momentum register. Used for block encodiding kinetic energy operator.
 - `mu`: A register used for implementing nested boxes for the momentum state preparation.
 - `nu_x`: x component of the momentum register for Coulomb potential.
 - `nu_y`: y component of the momentum register for Coulomb potential.
 - `nu_z`: z component of the momentum register for Coulomb potential.
 - `m`: an ancilla register in a uniform superposition.
 - `l`: The register for selecting the nuclei.
 - `sys`: The system register. Will store $\eta$ registers (x, y and z) compents of size num_bits_p. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). 


In [ ]:
signature, model = cases["select"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)

## `PrepareTFirstQuantization`
PREPARE for the kinetic energy operator for the first quantized chemistry Hamiltonian.

This prepares the state

$$
    |+\rangle\sum_{j=1}^{\eta}|j\rangle\sum_{w=0}^{2}|w\rangle
    \sum_{r=0}^{n_{p}-2}2^{r/2}|r\rangle
    \sum_{s=0}^{n_{p}-2}2^{s/2}|s\rangle
$$

We assume that the uniform superposition over ($i$ and) $j$ has already occured via
UniformSuperPositionIJFirstQuantization.

#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons.
 - `num_bits_rot_aa`: The number of bits of precision for the single qubit rotation for amplitude amplification. Called $b_r$ in the reference. 

#### Registers
 - `w`: a register to index one of three components of the momenta.
 - `r`: a register encoding bits for each component of the momenta.
 - `s`: a register encoding bits for each component of the momenta. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). page 19, section B


In [ ]:
signature, model = cases["prepare_t"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)

## `PrepareUVFirstQuantization`
PREPARE the U and V parts of the Hamiltonian.

#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons.
 - `num_atoms`: The number of atoms. $L$ in the reference.
 - `m_param`: $\mathcal{M}$ in the reference.
 - `lambda_zeta`: sum of nuclear charges.
 - `num_bits_nuc_pos`: The number of bits of precision for representing the nuclear coordinates. 

#### Registers
 - `mu`: The state controlling the nested boxes procedure.
 - `nu`: The momentum transfer register.
 - `m`: an ancilla register in a uniform superposition.
 - `l`: The register for atomic species.
 - `flag_nu`: Flag for success of the state preparation. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). page 19, section B


In [ ]:
signature, model = cases["prepare_uv"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)

## `SelectTFirstQuantization`
SELECT for the kinetic energy operator for the first quantized chemistry Hamiltonian.

#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons. 

#### Registers
 - `sys`: The system register.
 - `plus`: A $|+\rangle$ state.
 - `flag_T`: a flag to control on the success of the $T$ state preparation. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). page 20, section B


In [ ]:
signature, model = cases["select_t"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)

## `SelectUVFirstQuantization`
SELECT for the U and V operators for the first quantized chemistry Hamiltonian.

This does not include the controlled swaps from p_i and q_j system registers
into ancilla registers and back again. Hence there is no system register.

#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons.
 - `num_atoms`: The number of atoms.
 - `num_bits_nuc_pos`: The number of bits to store each component of the nuclear positions. $n_R$ in the reference. 

Registers:

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). 


In [ ]:
signature, model = cases["select_uv"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)

## `PrepareZetaState`
PREPARE the superpostion over $l$ weighted by $\zeta_l$.

See https://github.com/quantumlib/Qualtran/issues/473.
#### Parameters
 - `num_bits_p`: The number of bits to represent each dimension of the momentum register.
 - `eta`: The number of electrons.
 - `m_param`: $\mathcal{M}$ in the reference.
 - `lambda_zeta`: sum of nuclear charges. 

#### Registers
 - `l`: the register indexing the atomic number. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). page 23-24, last 3 paragraphs.


In [ ]:
signature, model = cases["zeta"]

### Example Instances

In [ ]:
print(model.name, "declared ports", signature.n_qubits())

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([(model.name, signature, {})])))

### Call Graph

In [ ]:
graph = modeled_call_graph(model, max_depth=1)
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print("Depth-one calls", [(child.name, count) for child, count in graph.leaf_counts.items()])
report = estimate_ftqc_resources(model)
assert report.emitted is None
print("Logical gate estimate", asdict(report.logical.gates))
print("Peak logical qubits", report.logical.peak_logical_qubits)